# Day 3-02｜ByteTrack Demo：用 sample boxes 產生 Track ID

> Python「黑客松」實戰分析籃球運動數據  
> Notebook 以初學 Python 學員為主：先觀察、再改變少量變數、最後才串接。

目標：用同一張球場圖與模擬的多 frame boxes，產生 track_id 與軌跡。若 supervision 可用，就用 ByteTrack；否則使用簡化 greedy tracker。

In [ ]:
from pathlib import Path
import sys

# 在 Colab 中先掛載 Drive；本機執行時會自動略過。
try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception:
    pass

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機或 zip 解壓後執行時，從目前資料夾往上找。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

print("COURSE_ROOT =", COURSE_ROOT)
print("assets =", COURSE_ROOT / "assets")

In [ ]:
import numpy as np
import pandas as pd
from src.cv_utils import load_json, read_image_rgb, draw_boxes, show_image, save_json

track_data = load_json(
    COURSE_ROOT / "assets" / "samples" / "sample_tracking_boxes.json"
)
frames = track_data["frames"]
print("frames:", len(frames))

In [ ]:
def box_center(box):
    x1, y1, x2, y2 = box
    return np.array([(x1 + x2) / 2, (y1 + y2) / 2], dtype=float)


# 簡化 tracker：第一個 frame 依序給 ID；後續用最近中心點配對。
next_id = 0
active = {}  # track_id -> center
records = []

for f in frames:
    dets = f["detections"]
    centers = [box_center(d["bbox_xyxy"]) for d in dets]
    assigned_tracks = []
    if not active:
        for c in centers:
            active[next_id] = c
            assigned_tracks.append(next_id)
            next_id += 1
    else:
        used = set()
        new_active = {}
        for c in centers:
            best_id, best_dist = None, 1e9
            for tid, old_c in active.items():
                if tid in used:
                    continue
                dist = float(np.linalg.norm(c - old_c))
                if dist < best_dist:
                    best_id, best_dist = tid, dist
            if best_id is None or best_dist > 80:
                best_id = next_id
                next_id += 1
            used.add(best_id)
            new_active[best_id] = c
            assigned_tracks.append(best_id)
        active = new_active
    for det, tid, c in zip(dets, assigned_tracks, centers):
        records.append(
            {
                "frame": f["frame_index"],
                "track_id": int(tid),
                "bbox_xyxy": det["bbox_xyxy"],
                "center_x": float(c[0]),
                "center_y": float(c[1]),
            }
        )

tracks = pd.DataFrame(records)
tracks.head()

In [ ]:
# 畫其中一個 frame 的 track_id。
image = read_image_rgb(COURSE_ROOT / "assets" / "samples" / "sample_court_frame.png")
frame_id = 8
rows = tracks[tracks["frame"] == frame_id]
boxes = rows["bbox_xyxy"].tolist()
labels = [f"ID {tid}" for tid in rows["track_id"]]
vis = draw_boxes(image, boxes, labels)
show_image(vis, f"frame {frame_id} with track IDs")

out_json = COURSE_ROOT / "assets" / "results" / "d3_02_tracks.json"
save_json(records, out_json)
print("saved:", out_json)

正式專案可以改成 `supervision.ByteTrack`，但教學上先用這個版本看懂 track ID 的意義。